# Career Path Analysis

Analyses the RFFM youth football ladder:
- What distinguishes top-tier players (SUPERLIGA/LIGA NACIONAL) from others in the same age group?
- How do players move up the ladder — with their team (promotion) or via individual transfer?
- How stable are squad rosters? How much do teams mix?
- Who has "disappeared" from RFFM data (likely transferred to RFEF / national competitions)?
- Which lower-division players look statistically similar to top-tier players?

**Requires** `player_career.xlsx` built by `player_career.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

BASE = Path("../output/processed/rffm")
SEASONS = sorted(d.name for d in BASE.iterdir() if d.is_dir() and d.name[0].isdigit())
print("Seasons:", SEASONS)

## 0. Load career base

In [ ]:
career_raw = pd.read_excel(BASE / "player_career.xlsx", dtype={"player_id": str, "birth_year": str})

LIST_COLS = ["seasons_active", "categories", "divisions", "clubs", "teams", "competitions"]
for c in LIST_COLS:
    career_raw[c] = career_raw[c].apply(
        lambda v: [x for x in v.split(";") if x] if isinstance(v, str) and v else []
    )

career_raw["birth_year"] = pd.to_numeric(career_raw["birth_year"], errors="coerce")
print(f"Career base: {len(career_raw):,} players, {len(career_raw.columns)} columns")
career_raw.head(2)

## 1. Temporal participation table

Re-load per-season participation with `season`, `team_id`, `category_base`, `division_level`.
This is the foundation for all transition / progression analysis.

In [ ]:
PART_COLS = ["player_id", "season", "competition_id", "team_id", "club_name_raw"]
COMP_COLS = ["competition_id", "category_base", "division_level"]

part_frames = []
for s in SEASONS:
    pp = BASE / s / "player_competition_participation.csv"
    cp = BASE / s / "competitions.csv"
    if not pp.exists():
        continue
    part = pd.read_csv(pp, dtype=str, usecols=PART_COLS)
    if cp.exists():
        comps = pd.read_csv(cp, dtype=str, usecols=COMP_COLS)
        part = part.merge(comps, on="competition_id", how="left")
    else:
        part["category_base"] = pd.NA
        part["division_level"] = pd.NA
    part_frames.append(part)

part_temporal = pd.concat(part_frames, ignore_index=True)
print(f"part_temporal: {len(part_temporal):,} rows")
print("Distinct category_base:", sorted(part_temporal["category_base"].dropna().unique()))

In [ ]:
# Per-season player stats (goals, matches per season)
pss_frames = []
for s in SEASONS:
    p = BASE / s / "player_season_stats.csv"
    if p.exists():
        df = pd.read_csv(p, dtype=str,
            usecols=["player_id", "season", "goals_total", "matches_played",
                     "starter_appearances", "called_up", "is_goalkeeper"])
        pss_frames.append(df)

pss_temporal = pd.concat(pss_frames, ignore_index=True)
for c in ["goals_total", "matches_played", "starter_appearances", "called_up"]:
    pss_temporal[c] = pd.to_numeric(pss_temporal[c], errors="coerce").fillna(0).astype(int)
pss_temporal["is_goalkeeper"] = pss_temporal["is_goalkeeper"].str.lower().eq("true")
print(f"pss_temporal: {len(pss_temporal):,} rows")

## 2. Labels and division tier mapping

In [ ]:
# Tier numbers from DIVISIONS.md (lower = higher prestige)
TIER_MAP = {
    "SUPERLIGA": 1, "LIGA NACIONAL": 1,
    "DIVISION DE HONOR": 2,
    "PRIMERA DIVISION AUTONOMICA": 3,
    "PREFERENTE": 4,
    "SEGUNDA DIVISION B": 5, "TERCERA FEDERACION": 5,
    "PRIMERA": 6,
    "SEGUNDA": 7,
    "TERCERA": 8,
}

# Top-tier division per youth category
TOP_TIER_DIV = {
    "JUVENIL":    "LIGA NACIONAL",
    "CADETE":     "SUPERLIGA",
    "INFANTIL":   "SUPERLIGA",
    "ALEVIN":     "SUPERLIGA",
    "BENJAMIN":   "DIVISION DE HONOR",
    "PREBENJAMIN":"PRIMERA DIVISION AUTONOMICA",
}
YOUTH_CATS = ["PREBENJAMIN", "BENJAMIN", "ALEVIN", "INFANTIL", "CADETE", "JUVENIL"]
CAT_ORDER = {c: i for i, c in enumerate(YOUTH_CATS)}  # higher = older

part_temporal["division_tier"] = part_temporal["division_level"].map(TIER_MAP)
part_temporal["is_top_tier"] = (
    part_temporal.apply(
        lambda r: r["division_level"] == TOP_TIER_DIV.get(r["category_base"], "__none__"),
        axis=1
    )
)

In [ ]:
career = career_raw.copy()

# Highest youth category each player reached
def highest_youth_cat(cats):
    youth = [c for c in cats if c in CAT_ORDER]
    return max(youth, key=lambda c: CAT_ORDER[c]) if youth else None

career["highest_category"] = career["categories"].apply(highest_youth_cat)
career["last_active_season"] = career["seasons_active"].apply(lambda s: s[-1] if s else None)

# Which categories did this player reach top-tier in?
top_tier_by_player = (
    part_temporal[part_temporal["is_top_tier"] & part_temporal["category_base"].isin(YOUTH_CATS)]
    .groupby("player_id")["category_base"]
    .apply(lambda x: sorted(set(x), key=lambda c: CAT_ORDER.get(c, -1)))
    .rename("top_tier_categories")
)

career = career.set_index("player_id").join(top_tier_by_player).reset_index()
career["top_tier_categories"] = career["top_tier_categories"].apply(
    lambda v: v if isinstance(v, list) else []
)
career["ever_top_tier"] = career["top_tier_categories"].apply(bool)

print(f"Players ever in top tier: {career['ever_top_tier'].sum():,} "
      f"({career['ever_top_tier'].mean():.1%} of all)")
print("\nTop-tier players by highest category reached:")
print(career[career["ever_top_tier"]]["highest_category"].value_counts())

## 3. Top-tier vs. non-top-tier profile comparison

For each youth category: compare players who reached the top tier (SUPERLIGA / LIGA NACIONAL)
vs. those who never did, using lifetime stats.

In [ ]:
# Enrich career with per-category stats from part_temporal + pss_temporal
# For each player: stats in their best/most recent participation per youth category
# We use lifetime totals from career (already aggregated) but split by category type

# Build per-(player, category_base) stats
# Join part_temporal with pss_temporal on (player_id, season)
part_with_stats = part_temporal[
    part_temporal["category_base"].isin(YOUTH_CATS)
].merge(
    pss_temporal[["player_id", "season", "goals_total", "matches_played",
                  "starter_appearances", "called_up"]],
    on=["player_id", "season"], how="left"
)

# Best division reached per player per category
best_div_per_cat = (
    part_temporal[part_temporal["category_base"].isin(YOUTH_CATS)]
    .dropna(subset=["division_tier"])
    .groupby(["player_id", "category_base"])["division_tier"]
    .min()  # lowest tier number = highest division
    .reset_index()
    .rename(columns={"division_tier": "best_tier_in_cat"})
)

print(best_div_per_cat.head())

In [ ]:
# Comparison per category: top-tier vs. not
# Use career-level stats (lifetime) for players whose HIGHEST category is this one
# to avoid mixing stats from different age groups

COMPARE_CATS = ["ALEVIN", "INFANTIL", "CADETE", "JUVENIL"]
METRICS = ["matches_played", "goals_total", "goals_from_acta", "win_rate",
           "seasons_count", "captain_appearances"]

rows = []
for cat in COMPARE_CATS:
    sub = career[career["highest_category"] == cat].copy()
    sub["in_top_tier"] = sub["top_tier_categories"].apply(lambda cats: cat in cats)
    for metric in METRICS:
        top = sub[sub["in_top_tier"]][metric].dropna()
        other = sub[~sub["in_top_tier"]][metric].dropna()
        if len(top) < 5 or len(other) < 5:
            continue
        u_stat, p_val = stats.mannwhitneyu(top, other, alternative="two-sided")
        rows.append({
            "category": cat,
            "metric": metric,
            "top_tier_n": len(top),
            "top_tier_median": round(top.median(), 2),
            "other_n": len(other),
            "other_median": round(other.median(), 2),
            "ratio": round(top.median() / other.median(), 2) if other.median() > 0 else None,
            "p_value": round(p_val, 4),
        })

profile_comparison = pd.DataFrame(rows)
print(profile_comparison.to_string(index=False))

## 4a. Progression up the age ladder

For each player, build their sequence of (season, category_base) entries
and check: did they move up? How fast?

In [ ]:
# Best category per player per season (pick highest age group)
youth_only = part_temporal[part_temporal["category_base"].isin(YOUTH_CATS)].copy()
youth_only["cat_order"] = youth_only["category_base"].map(CAT_ORDER)

best_cat_per_season = (
    youth_only.groupby(["player_id", "season"])["cat_order"]
    .max()
    .reset_index()
)
best_cat_per_season["category_base"] = best_cat_per_season["cat_order"].map(
    {v: k for k, v in CAT_ORDER.items()}
)

# For each player, sort by season and compute transitions between consecutive seasons
best_cat_per_season = best_cat_per_season.sort_values(["player_id", "season"])
best_cat_per_season["prev_cat_order"] = best_cat_per_season.groupby("player_id")["cat_order"].shift(1)
best_cat_per_season["cat_progression"] = best_cat_per_season["cat_order"] - best_cat_per_season["prev_cat_order"]

# Only look at rows that have a previous season
transitions_cat = best_cat_per_season.dropna(subset=["prev_cat_order"]).copy()
transitions_cat["move_type"] = pd.cut(
    transitions_cat["cat_progression"],
    bins=[-99, -1, 0, 1, 99],
    labels=["dropped_category", "same_category", "moved_up_1", "skipped_category"]
)

print("Category transition distribution:")
print(transitions_cat["move_type"].value_counts().to_string())
print(f"\nTotal inter-season transitions observed: {len(transitions_cat):,}")

In [ ]:
# Time spent in each category: seasons per player per category
seasons_in_cat = (
    youth_only.groupby(["player_id", "category_base"])["season"]
    .nunique()
    .reset_index()
    .rename(columns={"season": "seasons_in_cat"})
)

# Tag whether player reached top tier in this category
seasons_in_cat = seasons_in_cat.merge(
    best_div_per_cat[["player_id", "category_base", "best_tier_in_cat"]],
    on=["player_id", "category_base"], how="left"
)
seasons_in_cat["in_top_tier"] = seasons_in_cat["best_tier_in_cat"] == 1

print("Seasons spent per category (median):")
print(
    seasons_in_cat.groupby(["category_base", "in_top_tier"])["seasons_in_cat"]
    .median()
    .unstack("in_top_tier")
    .rename(columns={False: "non_top_tier", True: "top_tier"})
    .reindex([c for c in YOUTH_CATS if c in seasons_in_cat["category_base"].unique()])
)

## 4b. Mechanism of division progression

When a player moves to a higher division, did they:
- **stay with the same team** (their club got promoted), or
- **transfer** to a higher-division team individually?

We track `team_id` changes between consecutive seasons.

In [ ]:
# Best division (lowest tier number) per player per season, keeping team_id
# When multiple teams in same season, pick the team associated with best division
youth_div = (
    youth_only.dropna(subset=["division_tier"])
    .sort_values(["player_id", "season", "division_tier"])
    .drop_duplicates(subset=["player_id", "season"], keep="first")  # keep best division row
    [["player_id", "season", "team_id", "category_base", "division_level", "division_tier"]]
    .sort_values(["player_id", "season"])
    .reset_index(drop=True)
)

# Shift to get previous season values
g = youth_div.groupby("player_id")
youth_div["prev_team_id"] = g["team_id"].shift(1)
youth_div["prev_division_tier"] = g["division_tier"].shift(1)
youth_div["prev_season"] = g["season"].shift(1)

# Only consecutive seasons (skip gaps > 1 season)
def season_gap(s1, s2):
    if pd.isna(s1) or pd.isna(s2):
        return None
    try:
        return int(s2[:4]) - int(s1[:4])
    except (ValueError, TypeError):
        return None

youth_div["season_gap"] = youth_div.apply(
    lambda r: season_gap(r["prev_season"], r["season"]), axis=1
)

moves = youth_div[
    (youth_div["season_gap"] == 1) & youth_div["prev_division_tier"].notna()
].copy()

moves["tier_change"] = moves["prev_division_tier"] - moves["division_tier"]  # positive = moved up
moves["team_changed"] = moves["team_id"] != moves["prev_team_id"]

def classify_move(row):
    tc = row["tier_change"]
    changed = row["team_changed"]
    if tc > 0 and not changed:  return "promotion_with_team"
    if tc > 0 and changed:      return "transfer_up"
    if tc == 0 and not changed: return "stayed"
    if tc == 0 and changed:     return "transfer_lateral"
    if tc < 0 and not changed:  return "relegated_with_team"
    if tc < 0 and changed:      return "transfer_down"
    return "other"

moves["move_type"] = moves.apply(classify_move, axis=1)

print("Division move types (all years, all youth categories):")
print(moves["move_type"].value_counts().to_string())

In [ ]:
# How did players who REACHED top-tier get there?
# Among moves that resulted in division_tier == 1, what was the move type?
top_tier_arrival = moves[moves["division_tier"] == 1]
print(f"\nTotal arrivals at top-tier division: {len(top_tier_arrival):,}")
print(top_tier_arrival["move_type"].value_counts().to_string())
print("\n% breakdown:")
print((top_tier_arrival["move_type"].value_counts(normalize=True) * 100).round(1).to_string())

## 4c. Squad stability — how much do rosters turn over?

For each team × season: what fraction of players were there the previous season?

In [ ]:
# Load match lineups to get actual squad members per team per season
# (more accurate than participation table for who actually played)
lineup_frames = []
for s in SEASONS:
    for f in sorted((BASE / s / "match_lineups").glob("*.csv")):
        # Only youth categories — skip SENIOR/VETERANOS/UNIVERSITARIO
        cat = f.stem.upper()
        if cat not in [c.upper() for c in YOUTH_CATS]:
            continue
        df = pd.read_csv(f, dtype=str, usecols=["match_id", "team_id", "player_id"])
        df["season"] = s
        lineup_frames.append(df)

lineups_youth = pd.concat(lineup_frames, ignore_index=True)
print(f"Youth lineup rows: {len(lineups_youth):,}")

# Unique player per team per season
squad = lineups_youth.groupby(["season", "team_id"])["player_id"].apply(set).reset_index()
squad.columns = ["season", "team_id", "players"]
print(f"Team-season combinations: {len(squad):,}")

In [ ]:
# Retention rate: fraction of prev season squad still present
squad_sorted = squad.sort_values(["team_id", "season"]).reset_index(drop=True)
squad_sorted["prev_players"] = squad_sorted.groupby("team_id")["players"].shift(1)
squad_sorted["prev_season"] = squad_sorted.groupby("team_id")["season"].shift(1)

squad_sorted["season_gap"] = squad_sorted.apply(
    lambda r: season_gap(r["prev_season"], r["season"]), axis=1
)

consecutive = squad_sorted[(squad_sorted["season_gap"] == 1) & squad_sorted["prev_players"].notna()].copy()

def retention(row):
    curr, prev = row["players"], row["prev_players"]
    if not prev:
        return np.nan
    return len(curr & prev) / len(prev)

consecutive["retention_rate"] = consecutive.apply(retention, axis=1)

print("Squad retention rate (fraction of last season's players still present):")
print(f"  Overall median: {consecutive['retention_rate'].median():.1%}")
print(f"  25th–75th pct:  {consecutive['retention_rate'].quantile(0.25):.1%} – {consecutive['retention_rate'].quantile(0.75):.1%}")

In [ ]:
# Retention rate by division tier: are top-tier squads more or less stable?
# Join division info for the current season's team
div_for_team = (
    youth_only.dropna(subset=["division_tier"])
    .groupby(["team_id", "season"])["division_tier"]
    .min()
    .reset_index()
)

consecutive_div = consecutive.merge(div_for_team, on=["team_id", "season"], how="left")

print("\nRetention rate by division tier (current season):")
print(
    consecutive_div.groupby("division_tier")["retention_rate"]
    .agg(["median", "count"])
    .round(3)
    .rename(columns={"median": "retention_median", "count": "n_team_seasons"})
    .to_string()
)

In [ ]:
# Where do players from top-tier teams go next season?
# Players who were in tier-1 team, then look at their next season's division
top_tier_players_by_season = (
    youth_only[youth_only["division_tier"] == 1]
    [["player_id", "season", "team_id"]]
    .drop_duplicates()
)

# Next season for each player
next_season_div = (
    youth_div[["player_id", "season", "division_tier"]]
    .rename(columns={"season": "next_season", "division_tier": "next_tier"})
)

def next_season_str(s):
    try:
        start = int(s[:4]) + 1
        return f"{start}-{start+1}"
    except (ValueError, TypeError):
        return None

top_tier_players_by_season["next_season"] = top_tier_players_by_season["season"].apply(next_season_str)
top_tier_exodus = top_tier_players_by_season.merge(
    next_season_div, on=["player_id", "next_season"], how="left"
)

print("Next-season tier for players who were in top-tier (tier 1) this season:")
print(top_tier_exodus["next_tier"].value_counts(dropna=False).head(10).to_string())
total_tt = len(top_tier_exodus)
disappeared_tt = top_tier_exodus["next_tier"].isna().sum()
print(f"\nGone from RFFM next season: {disappeared_tt:,} ({disappeared_tt/total_tt:.1%})")

## 4d. Teammate quality effect

Hypothesis: playing alongside future top-tier players is a signal of talent pipeline quality.
We compute, for each player in each season, what fraction of their teammates
later appeared in a top-tier division.

In [ ]:
# Players who EVER reached top tier
ever_top_set = set(career[career["ever_top_tier"]]["player_id"])

# For each (season, team_id): fraction of registered players who ever reached top tier
team_roster = (
    youth_only.groupby(["season", "team_id"])["player_id"]
    .apply(set)
    .reset_index()
    .rename(columns={"player_id": "roster"})
)

team_roster["top_tier_fraction"] = team_roster["roster"].apply(
    lambda r: sum(1 for p in r if p in ever_top_set) / len(r) if r else 0.0
)

# Expand: each player gets their team's top_tier_fraction for that season
# (includes themselves — negligible self-inflation for squads of 15+)
roster_expanded = team_roster.explode("roster").rename(columns={"roster": "player_id"})

player_teammate_quality = (
    roster_expanded.groupby("player_id")["top_tier_fraction"]
    .mean()
    .rename("avg_teammate_top_tier_frac")
)

career = career.set_index("player_id").join(player_teammate_quality).reset_index()

# Compare teammate quality for players who did vs didn't reach top tier
print("Avg fraction of teammates who reached top tier:")
print(career.groupby("ever_top_tier")["avg_teammate_top_tier_frac"].describe().round(3))

## 5. Disappeared players — RFEF transfer hypothesis

Players who:
- Were born ≤ 2007 (at least 17-18 years old by 2025-2026)
- Reached CADETE or JUVENIL in RFFM
- Had NO activity in RFFM after 2023-2024 season

These players should be playing adult football somewhere — if they're not in RFFM,
the likely explanation is they transferred to RFEF (national federation) competitions
not tracked in this dataset.

In [ ]:
disappeared = career[
    (career["birth_year"] <= 2007) &
    (career["highest_category"].isin(["CADETE", "JUVENIL"])) &
    (career["last_active_season"] <= "2023-2024")
].copy()

print(f"Total disappeared players: {len(disappeared):,}")
print("\nBy highest category:")
print(disappeared["highest_category"].value_counts().to_string())
print("\nBy last active season:")
print(disappeared["last_active_season"].value_counts().sort_index().to_string())
print("\nBy birth year:")
print(disappeared["birth_year"].value_counts().sort_index().to_string())

In [ ]:
# High-confidence RFEF candidates: were in top-tier division before disappearing
top_tier_disappeared = disappeared[disappeared["ever_top_tier"]].copy()
print(f"Disappeared AND were in top-tier division: {len(top_tier_disappeared):,}")

# Top-30 by goals scored (most talented by raw output)
cols_show = ["player_id", "player_name", "birth_year", "last_active_season",
             "highest_category", "top_tier_categories", "goals_total",
             "matches_played", "win_rate", "clubs"]
top30_disappeared = (
    top_tier_disappeared.nlargest(30, "goals_total")
    [cols_show]
)
print("\nTop-30 disappeared top-tier players by goals:")
print(top30_disappeared.to_string(index=False))

## 6. Lookalikes — lower-division players with top-tier profiles

Find players currently active (2024-2025 or 2025-2026) who play in lower divisions
but whose stats match top-tier players of the same age group.

In [ ]:
# Currently active youth players who never reached top tier
currently_active = career[
    (career["last_active_season"] >= "2024-2025") &
    (career["highest_category"].isin(YOUTH_CATS)) &
    (~career["ever_top_tier"])
].copy()

currently_active["gpm"] = np.where(
    currently_active["matches_played"] > 0,
    currently_active["goals_from_acta"] / currently_active["matches_played"],
    0
)
currently_active["starter_rate"] = np.where(
    currently_active["called_up"] > 0,
    currently_active["starter_appearances"] / currently_active["called_up"],
    0
)
currently_active["captain_rate"] = np.where(
    currently_active["matches_played"] > 0,
    currently_active["captain_appearances"] / currently_active["matches_played"],
    0
)

print(f"Currently active non-top-tier youth players: {len(currently_active):,}")

In [ ]:
# Top-tier player profile per highest_category (median of each metric)
top_tier_active = career[
    (career["last_active_season"] >= "2024-2025") &
    (career["ever_top_tier"]) &
    (career["matches_played"] >= 10)
].copy()

top_tier_active["gpm"] = np.where(
    top_tier_active["matches_played"] > 0,
    top_tier_active["goals_from_acta"] / top_tier_active["matches_played"],
    0
)
top_tier_active["starter_rate"] = np.where(
    top_tier_active["called_up"] > 0,
    top_tier_active["starter_appearances"] / top_tier_active["called_up"],
    0
)
top_tier_active["captain_rate"] = np.where(
    top_tier_active["matches_played"] > 0,
    top_tier_active["captain_appearances"] / top_tier_active["matches_played"],
    0
)

SCORE_FEATURES = ["gpm", "starter_rate", "win_rate", "captain_rate"]

profile_by_cat = (
    top_tier_active.groupby("highest_category")[SCORE_FEATURES]
    .median()
)
print("Top-tier player profile (median per category):")
print(profile_by_cat.round(3))

In [ ]:
from sklearn.preprocessing import MinMaxScaler

lookalike_rows = []
for cat in COMPARE_CATS:
    if cat not in profile_by_cat.index:
        continue
    target = profile_by_cat.loc[cat]
    pool = currently_active[
        (currently_active["highest_category"] == cat) &
        (currently_active["matches_played"] >= 10)
    ].copy()
    if pool.empty:
        continue

    # Fill NaN win_rate with 0.5 (neutral)
    for f in SCORE_FEATURES:
        pool[f] = pool[f].fillna(0.5 if f == "win_rate" else 0)

    # Normalize both pool and target to [0,1] using pool's range
    scaler = MinMaxScaler()
    scaler.fit(pool[SCORE_FEATURES])
    pool_scaled = scaler.transform(pool[SCORE_FEATURES])
    target_vals = np.array([target.get(f, 0) for f in SCORE_FEATURES]).reshape(1, -1)
    target_scaled = scaler.transform(target_vals)[0]

    weights = np.array([0.35, 0.25, 0.25, 0.15])
    dists = np.sqrt(((pool_scaled - target_scaled) ** 2 * weights).sum(axis=1))
    pool["lookalike_dist"] = dists
    pool["lookalike_cat"] = cat
    lookalike_rows.append(pool)

lookalikes = pd.concat(lookalike_rows, ignore_index=True)
top30_lookalikes = (
    lookalikes.nsmallest(30, "lookalike_dist")
    [["player_name", "birth_year", "lookalike_cat", "lookalike_dist",
      "gpm", "starter_rate", "win_rate", "captain_rate",
      "matches_played", "last_active_season", "clubs"]]
)
print("Top-30 lookalikes (lower-division, statistically similar to top-tier profile):")
print(top30_lookalikes.to_string(index=False))

## 7. Export results

In [ ]:
def save(df, name):
    out = df.copy()
    # Serialize list columns
    for c in out.columns:
        if out[c].apply(lambda v: isinstance(v, (list, set))).any():
            out[c] = out[c].apply(lambda v: ";".join(str(x) for x in v) if isinstance(v, (list, set)) else v)
    path = BASE / f"career_analysis_{name}.csv"
    out.to_csv(path, index=False)
    print(f"Saved {len(out):,} rows → {path}")

save(profile_comparison, "top_tier_profile")
save(top_tier_disappeared, "disappeared")
save(top30_lookalikes, "lookalikes")

In [ ]:
# Sanity checks
print("=== SANITY CHECKS ===")
print(f"ever_top_tier rate: {career['ever_top_tier'].mean():.1%}")
print(f"Youngest 'disappeared' birth year: {disappeared['birth_year'].max()}")
print(f"  (should be ≤ 2007, got: {disappeared['birth_year'].max()})")

# Top-3 lookalikes spot-check
print("\nTop-3 lookalikes (manual spot-check):")
print(top30_lookalikes.head(3)[["player_name", "birth_year", "lookalike_cat",
                                "last_active_season", "clubs"]].to_string(index=False))

## 8. Squad career trace — Liga Nacional Juvenil (Real Madrid B)

Pick a top-tier JUVENIL team and trace every player's full RFFM career:
- When did they first appear in RFFM, in which category/division/club?
- How fast did they progress up the ladder?
- What were their stats just before joining the Liga Nacional squad?

Change `TARGET_TEAM_ID` to explore other teams:
| team_id | Team |
|---|---|
| `"5"` | REAL MADRID C.F. 'B' — best record (141W/5L over 5 seasons) |
| `"21"` | CLUB ATLETICO DE MADRID S.A.D. 'B' — consistent 2nd |
| `"96"` | C.D. LEGANES S.A.D. 'B' — trending up |
| `"563"` | GETAFE C.F. S.A.D. 'B' |
| `"60"` | RAYO VALLECANO DE MADRID S.A.D. 'B' |

In [27]:
TARGET_TEAM_ID = "5"  # Real Madrid B Juvenil

# ── 1. Get all players who ever appeared in this team (from participation)
squad_players = (
    part_temporal[part_temporal["team_id"] == TARGET_TEAM_ID]["player_id"]
    .unique()
    .tolist()
)
print(f"Unique players ever in team {TARGET_TEAM_ID}: {len(squad_players)}")

# ── 2. Full career timeline for these players: every (player_id, season, category, division, club)
timeline = (
    part_temporal[part_temporal["player_id"].isin(squad_players)]
    .merge(
        career[["player_id", "player_name", "birth_year"]],
        on="player_id", how="left"
    )
    .sort_values(["player_id", "season"])
)

# Add per-season stats
timeline = timeline.merge(
    pss_temporal[["player_id", "season", "goals_total", "matches_played",
                  "starter_appearances", "called_up", "is_goalkeeper"]],
    on=["player_id", "season"], how="left"
)

# Flag rows where this player was in the target team
timeline["in_target_team"] = timeline["team_id"] == TARGET_TEAM_ID

# Season they first appeared in target team
first_in_target = (
    timeline[timeline["in_target_team"]]
    .groupby("player_id")["season"]
    .min()
    .rename("first_target_season")
)
timeline = timeline.merge(first_in_target, on="player_id", how="left")

# Season they first appeared in RFFM at all
first_ever = (
    timeline.groupby("player_id")["season"]
    .min()
    .rename("first_rffm_season")
)
timeline = timeline.merge(first_ever, on="player_id", how="left")

# Seasons from first RFFM appearance to first target team appearance
def season_diff(s1, s2):
    try:
        return int(s2[:4]) - int(s1[:4])
    except (TypeError, ValueError):
        return None

timeline["years_to_target"] = timeline.apply(
    lambda r: season_diff(r["first_rffm_season"], r["first_target_season"]), axis=1
)

timeline[["player_name", "season", "category_base", "division_level",
                "club_name_raw", "in_target_team"]].head(10)

Unique players ever in team 5: 226


,player_name,season,category_base,division_level,club_name_raw,in_target_team
0,"ARROYO SANCHEZ, DIEGO",2018-2019,INFANTIL,DIVISION DE HONOR,REAL MADRID C.F.,False
1,"ARROYO SANCHEZ, DIEGO",2019-2020,INFANTIL,DIVISION DE HONOR,REAL MADRID C.F.,False
2,"ARROYO SANCHEZ, DIEGO",2020-2021,CADETE,PRIMERA DIVISION AUTONOMICA,REAL MADRID C.F.,False
3,"ARROYO SANCHEZ, DIEGO",2021-2022,CADETE,SUPERLIGA,REAL MADRID C.F.,False
4,"ARROYO SANCHEZ, DIEGO",2022-2023,JUVENIL,PRIMERA DIVISION AUTONOMICA,REAL MADRID C.F.,False
5,"ARROYO SANCHEZ, DIEGO",2023-2024,JUVENIL,LIGA NACIONAL,REAL MADRID C.F.,True
6,"RAMON NAVEROS, JACOBO",2018-2019,INFANTIL,DIVISION DE HONOR,REAL MADRID C.F.,False
7,"RAMON NAVEROS, JACOBO",2018-2019,INFANTIL,DIVISION DE HONOR,REAL MADRID C.F.,False
8,"RAMON NAVEROS, JACOBO",2019-2020,CADETE,PRIMERA DIVISION AUTONOMICA,REAL MADRID C.F.,False
9,"RAMON NAVEROS, JACOBO",2020-2021,CADETE,PRIMERA DIVISION AUTONOMICA,REAL MADRID C.F.,False


In [28]:
# ── 3. Build one row per player for the summary table
# Keep best (highest category) row per player per season,
# then build a compact per-player summary

# Category order numeric
timeline["cat_order"] = timeline["category_base"].map(CAT_ORDER)
timeline["div_tier"] = timeline["division_level"].map(TIER_MAP)

# Per player: best category/division in each season they were NOT yet in target team
pre_target = timeline[~timeline["in_target_team"]].copy()

# Aggregate: first RFFM season, first category, categories visited before target,
# best division before target, goals/matches before target
pre_summary = pre_target.groupby("player_id").agg(
    first_rffm_season=("first_rffm_season", "first"),
    first_target_season=("first_target_season", "first"),
    years_to_target=("years_to_target", "first"),
    categories_before=("category_base", lambda x: sorted(set(x.dropna()), key=lambda c: CAT_ORDER.get(c, -1))),
    best_div_before=("div_tier", "min"),
    goals_before=("goals_total", "sum"),
    matches_before=("matches_played", "sum"),
    is_gk=("is_goalkeeper", "any"),
).reset_index()

pre_summary["best_div_before_label"] = pre_summary["best_div_before"].map(
    {v: k for k, v in TIER_MAP.items() if isinstance(v, int)}
)

# Merge player names/birth years
pre_summary = pre_summary.merge(
    career[["player_id", "player_name", "birth_year"]],
    on="player_id", how="left"
)

# Players who came directly to target team (no prior RFFM record)
direct_arrivals = set(squad_players) - set(pre_summary["player_id"])
print(f"Players with RFFM history BEFORE target team: {len(pre_summary)}")
print(f"Players who appeared in target team with NO prior RFFM record: {len(direct_arrivals)}")

print("\nYears to reach target team (from first RFFM appearance):")
print(pre_summary["years_to_target"].value_counts().sort_index().to_string())

print("\nBest division reached BEFORE target team:")
print(pre_summary["best_div_before_label"].value_counts().to_string())

print("\nCategories played before reaching target team:")
cats_flat = pre_summary["categories_before"].explode()
print(cats_flat.value_counts().to_string())

Players with RFFM history BEFORE target team: 190
Players who appeared in target team with NO prior RFFM record: 36

Years to reach target team (from first RFFM appearance):
years_to_target
0    20
1    43
2    28
3    37
4    13
5    21
6    13
7    15

Best division reached BEFORE target team:
best_div_before_label
LIGA NACIONAL                  88
PRIMERA DIVISION AUTONOMICA    80
DIVISION DE HONOR              20
SEGUNDA                         1
PREFERENTE                      1

Categories played before reaching target team:
categories_before
JUVENIL       179
CADETE        147
INFANTIL       82
ALEVIN         47
OTHER          19
BENJAMIN       14
AFICIONADO     13


In [29]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── 4. Plotly timeline chart: one row per player, seasons on x-axis
# Each segment = one season; color = division_level; shape = category_base
# Thick border on target-team seasons

# Reduce to one row per (player, season) — best category/division for that season
best_per_season = (
    timeline
    .sort_values(["player_id", "season", "cat_order", "div_tier"])
    .drop_duplicates(subset=["player_id", "season"], keep="first")
    .copy()
)

# Determine display label for each player
best_per_season["display_name"] = (
    best_per_season["player_name"].str.split(",").str[0].str.strip()
    + " (" + best_per_season["birth_year"].fillna(0).astype(int).astype(str) + ")"
)

# Sort players by: first_target_season desc, then birth_year
player_order = (
    best_per_season[best_per_season["in_target_team"]]
    .groupby(["player_id", "display_name"])[["first_target_season", "birth_year"]]
    .first()
    .reset_index()
    .sort_values(["first_target_season", "birth_year"], ascending=[False, True])
)

# Fallback: players who only appear in target team
all_players_in_team = timeline["player_id"].unique()
missing = [p for p in all_players_in_team if p not in player_order["player_id"].values]
if missing:
    extra = (
        best_per_season[best_per_season["player_id"].isin(missing)]
        .groupby(["player_id", "display_name"])[["first_target_season", "birth_year"]]
        .first()
        .reset_index()
    )
    player_order = pd.concat([player_order, extra], ignore_index=True)

player_order["y_pos"] = range(len(player_order))
player_id_to_y = dict(zip(player_order["player_id"], player_order["y_pos"]))
player_id_to_name = dict(zip(player_order["player_id"], player_order["display_name"]))

# Division color palette
DIV_COLORS = {
    "LIGA NACIONAL":              "#1a1a2e",
    "SUPERLIGA":                  "#16213e",
    "DIVISION DE HONOR":          "#e94560",
    "PRIMERA DIVISION AUTONOMICA":"#f5a623",
    "PREFERENTE":                 "#7ed6a3",
    "PRIMERA":                    "#a0c4ff",
    "SEGUNDA":                    "#c9b8f4",
    "TERCERA":                    "#d3d3d3",
    "OTHER":                      "#eeeeee",
}

CAT_SYMBOLS = {
    "PREBENJAMIN": "circle",
    "BENJAMIN":    "square",
    "ALEVIN":      "diamond",
    "INFANTIL":    "triangle-up",
    "CADETE":      "star",
    "JUVENIL":     "hexagon",
}

# Build bar segments
all_seasons = sorted(best_per_season["season"].dropna().unique())
season_to_x = {s: i for i, s in enumerate(all_seasons)}

fig = go.Figure()

# Add one scatter trace per division level (for legend grouping)
added_divs = set()
added_cats = set()

for _, row in best_per_season.iterrows():
    pid = row["player_id"]
    if pid not in player_id_to_y:
        continue
    y = player_id_to_y[pid]
    x = season_to_x.get(row["season"])
    if x is None:
        continue

    div = row["division_level"] if pd.notna(row["division_level"]) else "OTHER"
    cat = row["category_base"] if pd.notna(row["category_base"]) else "OTHER"
    color = DIV_COLORS.get(div, "#eeeeee")
    symbol = CAT_SYMBOLS.get(cat, "circle")
    is_target = row["in_target_team"]

    hover = (
        f"<b>{player_id_to_name.get(pid, pid)}</b><br>"
        f"Season: {row['season']}<br>"
        f"Category: {cat}<br>"
        f"Division: {div}<br>"
        f"Club: {row.get('club_name_raw','')}<br>"
        f"Goals: {int(row['goals_total']) if pd.notna(row['goals_total']) else '?'}<br>"
        f"Matches: {int(row['matches_played']) if pd.notna(row['matches_played']) else '?'}"
    )

    fig.add_trace(go.Scatter(
        x=[x],
        y=[y],
        mode="markers",
        marker=dict(
            size=18 if is_target else 13,
            color=color,
            symbol=symbol,
            line=dict(color="#ffffff" if is_target else "rgba(0,0,0,0.2)",
                      width=3 if is_target else 0.5),
        ),
        hovertemplate=hover + "<extra></extra>",
        showlegend=False,
        name=div,
    ))

# Add connecting lines per player
for pid in player_id_to_y:
    rows_p = best_per_season[best_per_season["player_id"] == pid].sort_values("season")
    if len(rows_p) < 2:
        continue
    xs = [season_to_x[s] for s in rows_p["season"] if s in season_to_x]
    ys = [player_id_to_y[pid]] * len(xs)
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode="lines",
        line=dict(color="rgba(150,150,150,0.35)", width=1.5),
        showlegend=False,
        hoverinfo="skip",
    ))

# Shade the seasons where each player was in the target team
for pid, y in player_id_to_y.items():
    target_seasons = best_per_season[
        (best_per_season["player_id"] == pid) & best_per_season["in_target_team"]
    ]["season"].tolist()
    for s in target_seasons:
        x = season_to_x.get(s)
        if x is None:
            continue
        fig.add_shape(
            type="rect",
            x0=x - 0.4, x1=x + 0.4,
            y0=y - 0.45, y1=y + 0.45,
            fillcolor="rgba(26,26,46,0.08)",
            line=dict(color="rgba(26,26,46,0.3)", width=1),
        )

# Legend: division colors as colored squares
for div, color in DIV_COLORS.items():
    if div == "OTHER":
        continue
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=12, color=color, symbol="square"),
        name=div,
        showlegend=True,
    ))

# Legend: category shapes (grey, varying symbol)
for cat, sym in CAT_SYMBOLS.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=12, color="#888", symbol=sym),
        name=cat,
        showlegend=True,
    ))

fig.update_layout(
    title=dict(
        text=f"Career paths of players in team {TARGET_TEAM_ID}<br>"
             "<sup>Colour = division level · Shape = age category · "
             "Large/white border = season in target team</sup>",
        font_size=15,
    ),
    xaxis=dict(
        tickvals=list(season_to_x.values()),
        ticktext=list(season_to_x.keys()),
        tickangle=-40,
        title="Season",
        gridcolor="rgba(200,200,200,0.3)",
    ),
    yaxis=dict(
        tickvals=list(player_id_to_y.values()),
        ticktext=[player_id_to_name[pid] for pid in player_id_to_y],
        tickfont=dict(size=10),
        title="",
        autorange="reversed",
    ),
    height=max(500, len(player_id_to_y) * 26 + 120),
    width=1100,
    legend=dict(
        orientation="v",
        x=1.01, y=1,
        font_size=10,
        title_text="Division / Category",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=260, r=200, t=80, b=60),
)

fig.show()

In [34]:
# ── 5. Summary bar chart: how many seasons before reaching target team?
# + what was the best division they played in before arriving?

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Years of RFFM history before joining target team",
        "Best division reached BEFORE target team",
    ],
)

# Left: histogram of years_to_target
yt = pre_summary["years_to_target"].dropna().value_counts().sort_index()
fig2.add_trace(
    go.Bar(
        x=yt.index.astype(str),
        y=yt.values,
        marker_color="#e94560",
        text=yt.values,
        textposition="outside",
        name="Years to target",
    ),
    row=1, col=1,
)

# Right: best division before
div_cnt = pre_summary["best_div_before_label"].fillna("No data").value_counts()
div_order = ["LIGA NACIONAL", "SUPERLIGA", "DIVISION DE HONOR",
             "PRIMERA DIVISION AUTONOMICA", "PREFERENTE", "PRIMERA", "SEGUNDA", "No data"]
div_cnt = div_cnt.reindex([d for d in div_order if d in div_cnt.index]).dropna()
fig2.add_trace(
    go.Bar(
        x=div_cnt.index,
        y=div_cnt.values,
        marker_color=[DIV_COLORS.get(d, "#ccc") for d in div_cnt.index],
        text=div_cnt.values,
        textposition="outside",
        name="Best div before",
    ),
    row=1, col=2,
)

fig2.update_layout(
    title_text=f"Team {TARGET_TEAM_ID} — player backgrounds before joining",
    showlegend=False,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig2.update_xaxes(tickangle=-30, row=1, col=2)

fig2.show()

# Print summary table
print(f"\nSummary: {len(squad_players)} players total in team {TARGET_TEAM_ID}")
print(f"  With prior RFFM history:  {len(pre_summary)}")
print(f"  No prior RFFM record:     {len(direct_arrivals)}")
print(f"\nMedian years before joining: {pre_summary['years_to_target'].median():.1f}")
print(f"Median goals before joining: {pre_summary['goals_before'].median():.0f}")
print(f"Median matches before:       {pre_summary['matches_before'].median():.0f}")
print("\nTop 10 players by goals scored BEFORE joining target team:")
print(
    pre_summary.nlargest(10, "goals_before")
    [["player_name", "birth_year", "first_rffm_season", "first_target_season",
      "years_to_target", "categories_before", "best_div_before_label",
      "goals_before", "matches_before"]]
    .to_string(index=False)
)


Summary: 226 players total in team 5
  With prior RFFM history:  190
  No prior RFFM record:     36

Median years before joining: 3.0
Median goals before joining: 14
Median matches before:       91

Top 10 players by goals scored BEFORE joining target team:
                    player_name  birth_year first_rffm_season first_target_season  years_to_target                             categories_before best_div_before_label  goals_before  matches_before
 DIAZ MAROTO ALCAÑIZ, GUILLERMO      2009.0         2018-2019           2025-2026                7 [BENJAMIN, ALEVIN, INFANTIL, CADETE, JUVENIL]         LIGA NACIONAL           578             347
           MARTINEZ MENA, PABLO      2009.0         2018-2019           2025-2026                7 [BENJAMIN, ALEVIN, INFANTIL, CADETE, JUVENIL]         LIGA NACIONAL           427             401
        BARROSO PORTEROS, JAIME      2007.0         2018-2019           2023-2024                5           [ALEVIN, INFANTIL, CADETE, JUVENIL]      

## 8b. Homegrown vs. transfer — top JUVENIL teams compared

For each player in a top JUVENIL squad: was their very first RFFM appearance already
at the same club (homegrown / cantera), or did they come from somewhere else (transfer in)?

In [35]:
TOP_JUVENIL_TEAMS = {
    "5":   "Real Madrid B",
    "21":  "Atletico Madrid B",
    "96":  "Leganes B",
    "563": "Getafe B",
    "60":  "Rayo Vallecano B",
    "35":  "Mostoles URJC A",
    "46":  "Torrejon CF A",
}

# club_name_raw for each team_id — pull from participation table
team_club = (
    part_temporal[part_temporal["team_id"].isin(TOP_JUVENIL_TEAMS)]
    .groupby("team_id")["club_name_raw"]
    .agg(lambda x: x.mode().iat[0] if len(x) else "")
    .to_dict()
)

rows = []
for team_id, label in TOP_JUVENIL_TEAMS.items():
    club_name = team_club.get(team_id, "")

    # All players who appeared in this team
    players_in_team = (
        part_temporal[part_temporal["team_id"] == team_id]["player_id"]
        .unique()
        .tolist()
    )
    if not players_in_team:
        continue

    # For each player: their first RFFM appearance (season + club)
    first_appearance = (
        part_temporal[part_temporal["player_id"].isin(players_in_team)]
        .sort_values("season")
        .groupby("player_id")
        .first()
        .reset_index()
        [["player_id", "season", "club_name_raw"]]
        .rename(columns={"season": "first_season", "club_name_raw": "first_club"})
    )

    first_appearance["homegrown"] = first_appearance["first_club"] == club_name

    n_total      = len(first_appearance)
    n_homegrown  = first_appearance["homegrown"].sum()
    n_transfer   = n_total - n_homegrown

    # Among transfers: which clubs did they come from most?
    top_origins = (
        first_appearance[~first_appearance["homegrown"]]["first_club"]
        .value_counts()
        .head(3)
        .index.tolist()
    )

    rows.append({
        "team_id":      team_id,
        "team":         label,
        "club_name":    club_name,
        "total_players":n_total,
        "homegrown":    n_homegrown,
        "transferred_in": n_transfer,
        "pct_homegrown":  round(100 * n_homegrown / n_total, 1) if n_total else 0,
        "top_feeder_clubs": " / ".join(top_origins),
    })

hg = pd.DataFrame(rows).sort_values("pct_homegrown", ascending=False)
print(hg[["team", "total_players", "homegrown", "transferred_in",
          "pct_homegrown", "top_feeder_clubs"]].to_string(index=False))

# ── Plotly stacked bar
fig3 = go.Figure()

fig3.add_trace(go.Bar(
    name="Homegrown (cantera)",
    x=hg["team"],
    y=hg["homegrown"],
    marker_color="#1a1a2e",
    text=hg["homegrown"],
    textposition="inside",
    textfont=dict(color="white", size=12),
))
fig3.add_trace(go.Bar(
    name="Transferred in",
    x=hg["team"],
    y=hg["transferred_in"],
    marker_color="#e94560",
    text=hg["transferred_in"],
    textposition="inside",
    textfont=dict(color="white", size=12),
))

# % homegrown label on top
for _, r in hg.iterrows():
    fig3.add_annotation(
        x=r["team"],
        y=r["total_players"] + 1.5,
        text=f"{r['pct_homegrown']}% cantera",
        showarrow=False,
        font=dict(size=11, color="#333"),
    )

fig3.update_layout(
    barmode="stack",
    title="Homegrown vs. transferred-in players — top JUVENIL teams",
    xaxis_title="",
    yaxis_title="Players (all-time squad)",
    legend=dict(orientation="h", x=0.3, y=1.08),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=460,
    width=850,
)
fig3.show()

             team  total_players  homegrown  transferred_in  pct_homegrown                                                                    top_feeder_clubs
Atletico Madrid B            194        159              35           82.0     RAYO VALLECANO DE MADRID S.A.D. / ATLETICO MADRILEÑO C.F. / C.D. LEGANES S.A.D.
    Real Madrid B            226        181              45           80.1          RAYO VALLECANO DE MADRID S.A.D. / GETAFE C.F. S.A.D. / C.D. LEGANES S.A.D.
    Torrejon CF A            104         44              60           42.3       GETAFE C.F. S.A.D. / CLUB ATLETICO DE MADRID S.A.D. / ATLETICO MADRILEÑO C.F.
 Rayo Vallecano B            199         80             119           40.2              CLUB ATLETICO DE MADRID S.A.D. / GETAFE C.F. S.A.D. / REAL MADRID C.F.
        Leganes B            209         81             128           38.8 CLUB ATLETICO DE MADRID S.A.D. / REAL MADRID C.F. / RAYO VALLECANO DE MADRID S.A.D.
         Getafe B            216         82   